In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

print("--- APPROACH 1: RAW LINEAR REGRESSION ---")
df = pd.read_csv('combined_stock_data.csv')
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df['High'] = df.groupby('Company Name')['High'].ffill()
df['Low'] = df.groupby('Company Name')['Low'].ffill()
df['Volume'] = df['Volume'].fillna(0)

# Feature Engineering & Unified Target
df['Day_Range'] = df['High'] - df['Low']
df['Daily_Return'] = df.groupby('Company Name')['Close'].pct_change() * 100
df['Target_Trend'] = (df['Close'] > df['Open']).astype(int)

features = ['Open', 'High', 'Low', 'Volume', 'Day_Range', 'Daily_Return']
df_clean = df.dropna(subset=features + ['Target_Trend']).copy()

X = df_clean[features]
y = df_clean['Target_Trend']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

lin_model = LinearRegression()
lin_model.fit(X_train, y_train)
lin_preds = lin_model.predict(X_test)

print(f"Raw Output Sample: {lin_preds[:5]}")
print("Problem: Linear Regression outputs decimals (like 0.45 or 1.12) instead of a clean 0 or 1. It does not understand classification boundaries.")

--- APPROACH 1: RAW LINEAR REGRESSION ---
Raw Output Sample: [0.48403165 0.44916973 0.44862549 0.4951554  0.36930282]
Problem: Linear Regression outputs decimals (like 0.45 or 1.12) instead of a clean 0 or 1. It does not understand classification boundaries.


In [2]:
from sklearn.metrics import accuracy_score

print("\n--- APPROACH 2: FORCING A THRESHOLD ---")
print("We force any prediction above 0.5 to be 1 (Bullish) and below 0.5 to be 0 (Bearish).")

lin_preds_binary = (lin_preds >= 0.5).astype(int)
lin_accuracy = accuracy_score(y_test, lin_preds_binary)

print(f"Forced Linear Accuracy: {lin_accuracy * 100:.2f}%")
print("Conclusion: While we can force it to act like a classifier, it is mathematically clumsy. We need true Classification models.")


--- APPROACH 2: FORCING A THRESHOLD ---
We force any prediction above 0.5 to be 1 (Bullish) and below 0.5 to be 0 (Bearish).
Forced Linear Accuracy: 71.06%
Conclusion: While we can force it to act like a classifier, it is mathematically clumsy. We need true Classification models.
